# 금감원 바로 이 목소리 셀레니움 자동 다운로드 구현

In [ ]:
import os
import time
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import NoSuchElementException

# --- 설정 ---
# 영상 파일을 저장할 폴더 이름
DOWNLOAD_DIR = "videos"
# 접속할 URL
BASE_URL = ""

# --- 폴더 생성 ---
if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)
    print(f"'{DOWNLOAD_DIR}' 폴더를 생성했습니다.")

# --- 웹 드라이버 설정 ---
try:
    print("웹 드라이버를 설정합니다...")
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)
    print("웹 드라이버 설정 완료.")
except Exception as e:
    print(f"드라이버 설정 중 오류 발생: {e}")
    # 드라이버 설정 실패 시 여기서 중단
    exit()

# --- 메인 페이지에서 게시물 링크 수집 ---
video_page_urls = []
for i in range(1, 13):
    try:
        BASE_URL = f"https://www.fss.or.kr/fss/bbs/B0000203/list.do?menuNo=200686&bbsId=&cl1Cd=&pageIndex={i}&sdate=&edate=&searchCnd=1&searchWrd="
        print(url)    
        print(f"메인 페이지({BASE_URL})에 접속합니다...")
        driver.get(BASE_URL)
        time.sleep(3)  # 페이지 로딩 대기

        print("게시물 링크를 수집합니다...")
        # <div class="bd-list-thumb-a"> 안의 <a> 태그를 모두 찾음
        link_elements = driver.find_elements(By.CSS_SELECTOR, "div.bd-list-thumb-a a")
        for elem in link_elements:
            video_page_urls.append(elem.get_attribute("href"))

        print(f"총 {len(video_page_urls)}개의 게시물을 찾았습니다.")

    except Exception as e:
        print(f"메인 페이지 처리 중 오류 발생: {e}")

    # --- 각 게시물 페이지에 접속하여 영상 다운로드 ---
    for j, url in enumerate(video_page_urls):
        try:
            print(f"[{j+1}/{len(video_page_urls)}] 게시물에 접속합니다: {url}")
            driver.get(url)
            time.sleep(3)  # 페이지 로딩 대기

            # 영상 제목 가져오기 (<span class="name">)
            title_element = driver.find_element(By.CSS_SELECTOR, "span.name")
            video_title = title_element.text.strip()
            # 파일명으로 부적합한 문자 제거 및 공백을 밑줄로 변경
            safe_filename = ".".join(
                c for c in video_title if c.isalnum() or c in (" ", "_")
            ).replace(" ", "_")
            file_path = os.path.join(DOWNLOAD_DIR, f"{safe_filename}.mp4")

            # 이미 파일이 존재하면 건너뛰기
            # if os.path.exists(file_path):
            #     print(f"'{file_path}' 파일이 이미 존재하므로 건너뜁니다.")
            #     continue

            # video 태그의 src 속성 값(영상 URL) 가져오기
            video_element = driver.find_element(By.TAG_NAME, "video")
            video_src = video_element.get_attribute("src")

            if not video_src:
                print("영상 소스 URL을 찾을 수 없습니다.")
                continue

            # requests를 사용하여 영상 다운로드
            print(f"'{video_title}' 영상을 다운로드합니다...")
            response = requests.get(video_src, stream=True)

            if response.status_code == 200:
                with open(file_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"성공적으로 '{file_path}'에 저장했습니다.")
            else:
                print(f"다운로드 실패. HTTP 상태 코드: {response.status_code}")

        except NoSuchElementException as e:
            print(f"페이지에서 필요한 요소(제목 또는 비디오)를 찾지 못했습니다: {e}")
        except Exception as e:
            print(f"처리 중 오류 발생: {e}")

# --- 드라이버 종료 ---
print("모든 작업이 완료되었습니다. 브라우저를 종료합니다.")
driver.quit()

'videos' 폴더를 생성했습니다.
웹 드라이버를 설정합니다...
웹 드라이버 설정 완료.
https://www.fss.or.kr/fss/bbs/B0000203/view.do?nttId=18322&menuNo=200686&sdate=&edate=&searchCnd=1&searchWrd=&pageIndex=2
메인 페이지(https://www.fss.or.kr/fss/bbs/B0000203/list.do?menuNo=200686&bbsId=&cl1Cd=&pageIndex=1&sdate=&edate=&searchCnd=1&searchWrd=)에 접속합니다...
게시물 링크를 수집합니다...
총 8개의 게시물을 찾았습니다.
[1/8] 게시물에 접속합니다: https://www.fss.or.kr/fss/bbs/B0000203/view.do?nttId=130785&menuNo=200686&sdate=&edate=&searchCnd=1&searchWrd=&pageIndex=1
'금감원_보이스피싱_12_수정07.mp4.mp4 (파일크기: 18,748KB)' 영상을 다운로드합니다...
성공적으로 'videos\금.감.원._.보.이.스.피.싱._.1.2._.수.정.0.7.m.p.4.m.p.4._.파.일.크.기._.1.8.7.4.8.K.B.mp4'에 저장했습니다.
[2/8] 게시물에 접속합니다: https://www.fss.or.kr/fss/bbs/B0000203/view.do?nttId=130784&menuNo=200686&sdate=&edate=&searchCnd=1&searchWrd=&pageIndex=1
'금감원_보이스피싱_11_수정07.mp4.mp4 (파일크기: 25,798KB)' 영상을 다운로드합니다...
성공적으로 'videos\금.감.원._.보.이.스.피.싱._.1.1._.수.정.0.7.m.p.4.m.p.4._.파.일.크.기._.2.5.7.9.8.K.B.mp4'에 저장했습니다.
[3/8] 게시물에 접속합니다: https://www.fss.or.kr/fss/bbs

KeyboardInterrupt: 